<a href="https://colab.research.google.com/github/masaki-kawa/uts-study-notes/blob/main/data/raw/colab/Deep_Learning_Lab4_Exercise2_Solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transfer Learning and Fine-Tuning


---

In this second exercise, we will learn how to apply fine-tuning on a model which is just an extension of transfer learning.

This exercise is split in several parts:
1.   Loading and Exploring Dataset
2.   Preparing Dataset
3.   Defining the Architecture of VGG16
4.   Training and Evaluation of the Model
5.   Analysing the Results

# Predicting cats versus dogs (binary classification)

## Dataset

Same dataset as the first exercise.

## Objective

Our goal is to use a pre-trained Convolution Neural Network model and assess its performance then we will fine-tune it and see the impact.

## Instructions

This is a guided exercise where some of the code have already been pre-defined. Your task is to fill the remaining part of the code (it will be highlighted with placehoders) to train and evaluate your model.

This exercise is split in several parts:
1.   Loading and Exploration of the Dataset
2.   Preparing the Dataset
3.   Load a pre-trained VGG16 model
4.   Training and Evaluation of the VGG16 model

## Exercise 2 Solution

### 1. Loading and exploring the Dataset

**[1.1]** First we need to import the relevant class and libraries from PyTorch including 'os' in python. One method of utilizing operating system-dependent functionality in Python is through the OS module.The OS module provides functions that allow you to interact with underlying operating system that Python is running on – be that Windows, Mac or Linux.

In [ ]:
# Solution
import torch
import torchvision
import torchvision.transforms as transforms
from torchsummary import summary
import os

import random
import shutil
import zipfile
from pathlib import Path
from PIL import Image

**[1.2]**
Download the dataset and sample it.

##### Task: Download the dataset locally

In [ ]:
# Download and extract the dataset
random.seed(42)

file_url = "https://download.microsoft.com/download/3/e/1/3e1c3f21-ecdb-4869-8368-6deba77b919f/kagglecatsanddogs_5340.zip"

dataset_dir = Path("/content/cats_and_dogs_filtered")
zip_path = Path("/content/kagglecatsanddogs_5340.zip")
raw_dir = Path("/content/PetImages")

if not dataset_dir.exists():
    # Download
    if not zip_path.exists():
        !wget -O /content/kagglecatsanddogs_5340.zip "$file_url"

    # Extract
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content")

    # Create the expected structure
    for split in ["train", "validation"]:
        for cls in ["cats", "dogs"]:
            (dataset_dir / split / cls).mkdir(parents=True, exist_ok=True)

    # Keep only valid JPGs
    def valid_images(folder):
        files = []
        for p in sorted(folder.glob("*.jpg")):
            try:
                with Image.open(p) as img:
                    img.verify()
                files.append(p)
            except Exception:
                pass
        return files

    cats = valid_images(raw_dir / "Cat")
    dogs = valid_images(raw_dir / "Dog")

    # Sample only 1000 train + 500 validation per class
    random.shuffle(cats)
    random.shuffle(dogs)
    cats = cats[:1500]
    dogs = dogs[:1500]

    def copy_split(files, cls_name):
        train_files = files[:1000]
        val_files = files[1000:1500]

        for src in train_files:
            shutil.copy2(src, dataset_dir / "train" / cls_name / src.name)

        for src in val_files:
            shutil.copy2(src, dataset_dir / "validation" / cls_name / src.name)

    copy_split(cats, "cats")
    copy_split(dogs, "dogs")

print("Dataset ready at:", dataset_dir)
print("Train cats:", len(list((dataset_dir / "train" / "cats").glob("*.jpg"))))
print("Train dogs:", len(list((dataset_dir / "train" / "dogs").glob("*.jpg"))))
print("Val cats:", len(list((dataset_dir / "validation" / "cats").glob("*.jpg"))))
print("Val dogs:", len(list((dataset_dir / "validation" / "dogs").glob("*.jpg"))))

--2026-03-15 01:16:32--  https://download.microsoft.com/download/3/e/1/3e1c3f21-ecdb-4869-8368-6deba77b919f/kagglecatsanddogs_5340.zip
Resolving download.microsoft.com (download.microsoft.com)... 23.61.214.15, 2600:1406:5400:2ac::317f, 2600:1406:5400:2ae::317f
Connecting to download.microsoft.com (download.microsoft.com)|23.61.214.15|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 824887076 (787M) [application/octet-stream]
Saving to: ‘/content/kagglecatsanddogs_5340.zip’

/content/kagglecats 100%[===================>] 786.67M  69.8MB/s    in 11s     

2026-03-15 01:16:43 (69.7 MB/s) - ‘/content/kagglecatsanddogs_5340.zip’ saved [824887076/824887076]



/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Dataset ready at: /content/cats_and_dogs_filtered
Train cats: 1000
Train dogs: 1000
Val cats: 500
Val dogs: 500


### 2.   Preparing the Dataset

**[2.1]** Apply transformation using compose class. This time we will not only normalize the images but we will also perform some data transformation such as RandomResizedCrop(100) [ https://pytorch.org/vision/main/generated/torchvision.transforms.RandomResizedCrop.html ] and RandomHorizontalFlip [ https://pytorch.org/vision/main/generated/torchvision.transforms.RandomHorizontalFlip.html ] for train data and Resize (100) and CenterCrop(100) for test data. Data augmentation is a good practice for the train set. Here, we randomly crop the image to 100x100 and randomly flip it horizontally. After then we will normalizes the tenosr images of train and test set using the normalize function of torchvision [ https://pytorch.org/vision/main/generated/torchvision.transforms.Normalize.html ].

In [ ]:
# Solution
# Define transformations for training data
transform_train = transforms.Compose([
    transforms.RandomResizedCrop(100),                                            # Randomly crop and resize the image to 100x100
    transforms.RandomHorizontalFlip(),                                            # Randomly flip the image horizontally
    transforms.ToTensor(),                                                        # Convert the image to a PyTorch tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize the image
])

# Define transformations for test/validation data
transform_test = transforms.Compose([
    transforms.Resize(100),                                                       # Resize the image to 100x100
    transforms.CenterCrop(100),                                                   # Crop the center of the image to 100x100
    transforms.ToTensor(),                                                        # Convert the image to a PyTorch tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize the image
])

**[2.2]**
The images are already split into a 'train' and 'validation' folders under 'cats_and_dogs_filtered'

##### Task: Create 3 variables called train_dataset, test_dataset and full_dataset that will contain train, test data from the 'train' and 'validation' folders and combine the train and test data and save it into a variable called "full_dataset"

In [ ]:
# Solution
# Load training dataset
train_dataset = torchvision.datasets.ImageFolder(root=os.path.join(dataset_dir, 'train'), transform=transform_train)

# Load test dataset
test_dataset = torchvision.datasets.ImageFolder(root=os.path.join(dataset_dir, 'validation'), transform=transform_test)

# Combine datasets
full_dataset = torch.utils.data.ConcatDataset([train_dataset, test_dataset])

**[2.3]**
Lets split the size of the full_dataset into train, test and validation data set according to following ratio 70%:15%:15% and save it into train_size, val_size and test_size.

In [ ]:
# Solution
# Calculate sizes for training, validation, and test sets
total_data = len(full_dataset)
train_size = int(0.7 * total_data)
val_size = int(0.15 * total_data)
test_size = int(0.15 * total_data)

**[2.4]**
According to the size of the dataset, now we will split a dataset into several subsets such as train_dataset,val_dataset, and test_dataset by using the random_split function (https://pytorch.org/docs/stable/data.html)

In [ ]:
# Solution
# Split the dataset
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size, test_size])

**[2.5]**  Now we will call the DataLoader function that iteratively loads data based on batch size, shuffle and save it into three different variables called `train_loader`, `val_loader` and `test_loader`. Set the `BATCH_SIZE` to 20. The shuffle is a boolean variable. By default the shuffle value is false. If the shuffle is `True` means that the data is randomly shuffled before each epoch, so the order of the data is different in each epoch. (https://pytorch.org/tutorials/beginner/basics/data_tutorial.html)

In [ ]:
# Solution
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=20, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=20, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=20, shuffle=False)

**[2.6]**  Let's save the length of `train_dataset`, `val_dataset`, and `test_dataset` into `total_train`, `total_val` and `total_test`.

In [ ]:
# Solution
total_train = len(train_dataset)
total_val = len(val_dataset)
total_test = len(test_dataset)

**[2.7]**  Let's have a look at the length of `train_loader`, `val_loader`, and `test_loader`.

In [ ]:
# Solution
print(f"Size of train_loader: {len(train_loader)}")
print(f"Size of val_loader: {len(val_loader)}")
print(f"Size of test_loader: {len(test_loader)}")

Size of train_loader: 105
Size of val_loader: 23
Size of test_loader: 23


# 3: Defining the Architecture of VGG16

**[3.1]** Import `torch.nn` as `nn`, and to import VGG16 architecture we need to import `torchcision.models` as `models`

In [ ]:
# Solution
import torch.nn as nn
import torchvision.models as models

**[3.2]** Nwo we will initialize the VGG16 model by specifying the parameter `pretrained = True`, (https://pytorch.org/vision/0.12/generated/torchvision.models.vgg16.html).

In [ ]:
# Solution
vgg16 = models.vgg16(pretrained=True)  # Load pre-trained VGG16 model

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:06<00:00, 84.6MB/s]


**[3.4]** Let's have a look at the VGG16 architecture

##### Task: Print the summary of the VGG16 model

In [ ]:
# Solution
print(vgg16)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

**[3.3]** Let's freeze the first 12 layers of the model by setting the value of `requires_grad` as `false`, i.e. no changes happen to its parameters. (https://pytorch.org/docs/stable/generated/torch.Tensor.requires_grad.html#torch.Tensor.requires_grad)

In [ ]:
# Solution
# Freeze the first 12 layers of the model
frozen_layers = 12                                                # Number of layers to freeze
for idx, (name, param) in enumerate(vgg16.named_parameters()):    # Iterate through named parameters of the model
    if idx < frozen_layers:                                       # Check if the index is less than the number of frozen layers
        param.requires_grad = False                               # Set requires_grad to False to freeze the parameter

**[3.5]** As you can see, we have now a VGG16 model with no parameters update upto 12 layers and top layers used for making the predictions. We will add 2 fully connected layers to the VGG16.

##### Task: Create a new model by combining the VGG16 model to 2 fully connnected layers (the first one will have 500 units)

In [ ]:
# Solution
# Add additional layers to the model
num_features = vgg16.classifier[6].in_features    # Get the number of input features for the last layer
vgg16.classifier[6] = nn.Sequential(              # Replace the last layer with a new sequential layer
    nn.Linear(num_features, 500),                 # Add a linear layer with input size `num_features` and output size 500
    nn.ReLU(),                                    # Apply ReLU activation function
    nn.Linear(500, 1),                            # Add another linear layer with input size 500 and output size 1
    nn.Sigmoid()                                  # Apply sigmoid activation function to obtain probabilities for binary classification
)

**[3.6]** After modifying the last layer of VGG16, let's have a look at the modified VGG16 architecture

##### Task: Print the summary of newly modified VGG16 model

In [ ]:
# Solution
print(vgg16)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

### 4. Training and Evaluation of the Model

**[4.1]** Let's create a variable called `device` that will automatically select a GPU if available. Otherwise it will default to CPU.

In [ ]:
# # Solution
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

**[4.2]** Import the relevant class and libraries for `optim`,`ReduceLROnPlateau` and `SummaryWriter`

In [ ]:
# Solution
from torch import nn, optim
from torch.optim.lr_scheduler import ReduceLROnPlateau    # For adaptive learning rate adjustment
from torch.utils.tensorboard import SummaryWriter         # For logging training progress

**[4.3]**  Instantiate a `nn.BCELoss()` and save it into a variable called `criterion`. After then Instantiate a `torch.optim.Adam()` optimizer with the model's parameters and 0.0001 as learning rate and save it into a variable called `optimizer`

In [ ]:
# Solution
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(vgg16.parameters(), lr=0.0001)

**[4.4]**  Set up a learning rate scheduler `ReduceLROnPlateau` for reducing the learning rate when a metric has stopped improving [ https://pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.ReduceLROnPlateau.html ]

In [ ]:
# Solution
# Scheduler reduces learning rate when validation loss plateaus
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.2, patience=3, min_lr=0.0000001)

**[4.5]** Define the value of early stopping parameters such as `early_step_counter = 0`, `early_stop_patience = 5`

In [ ]:
# Solution
# Set counters and patience for early stopping (to avoid overfitting)
early_stop_counter = 0
early_stop_patience = 10      # Allow up to 10 epochs without improvement
best_val_loss = float('inf')  # Initialize best validation loss to infinity

**[4.6]** Lets introduce a function called `check_early_stop` to check for early stopping if the current validation losses do not improve based on the assigned value `early_stop_patience`.

In [ ]:
# Solution
# Define a function to check for early stopping
def check_early_stop(val_loss):
    global early_stop_counter, best_val_loss  # Access global variables for early stopping

    # Check if the current validation loss is better than the best validation loss seen so far
    if val_loss < best_val_loss:
        best_val_loss = val_loss  # Update the best validation loss
        early_stop_counter = 0    # Reset the counter if improvement is seen
    else:
        early_stop_counter += 1   # Increment the counter if no improvement

    # Check if the early stopping criterion (patience) has been met
    if early_stop_counter >= early_stop_patience:
        print("Early stopping triggered!")          # Print a message indicating early stopping
        return True                                 # Return True to indicate that early stopping is triggered
    else:
        return False                                # Return False to indicate that early stopping is not triggered

**[4.7]** Move newly designed model `vgg16` to device.

In [ ]:
# Solution
vgg16.to(device)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

**[4.8]** **Training:** Now it is time to train our model. Set the `EPOCHS` to 5 and create a for loop that will iterate based on the EPOCHS value. A nested loop is initiated that extracts images and labels from `train_loader` and introduce the following logics:
- reset the gradients (https://pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)
- perform the forward propagation and get the model predictions
- calculate the loss between the predictions and the actuals
- perform back propagation
- update the weights
- Count the total loss

To validate the model a nested loop is initiated that extracts images and labels from `val_loader` and introduce the following logics:
- disable computing gradients (https://pytorch.org/docs/stable/generated/torch.no_grad.html)
- perform the forward propagation and get the model predictions
- calculate the loss between the predictions and the actuals
- Count the total loss
- Count the correct outcome

In [ ]:
# Solution
# Initialize lists to store metrics
train_losses = []         # List to store training losses for each epoch
train_accuracies = []     # List to store training accuracies for each epoch
val_losses = []           # List to store validation losses for each epoch
val_accuracies = []       # List to store validation accuracies for each epoch

# Training loop
num_epochs = 15           # Number of epochs for training

for epoch in range(num_epochs): # Iterate over each epoch
    vgg16.train()               # Set the model to train mode
    running_loss = 0.0          # Initialize running loss for this epoch
    correct_train = 0           # Initialize number of correct predictions for training
    total_train = 0             # Initialize total number of samples for training

    # Iterate over the training dataset
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)   # Move inputs and labels to device
        optimizer.zero_grad()                                   # Zero the parameter gradients
        outputs = vgg16(inputs)                                 # Forward pass
        loss = criterion(outputs.squeeze(), labels.float())     # Compute the loss
        loss.backward()                                         # Backward pass
        optimizer.step()                                        # Weight updates
        running_loss += loss.item()                             # Count the total loss

        # Calculate training accuracy
        predicted = torch.round(outputs).squeeze()
        correct_train += (predicted == labels).sum().item()
        total_train += labels.size(0)                           # Count total samples

    train_loss = running_loss / len(train_loader)               # Calculate average training loss
    train_accuracy = correct_train / total_train                # Calculate training accuracy

    # Validation loop
    vgg16.eval()                # Set the model to evaluation mode
    val_running_loss = 0.0      # Initialize running loss for validation
    correct_val = 0             # Initialize number of correct predictions for validation
    total_val = 0               # Initialize total number of samples for validation

    # Disable gradient calculation for validation
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)     # Move inputs and labels to device
            outputs = vgg16(inputs)                                   # Forward pass
            loss = criterion(outputs.squeeze(), labels.float())       # Compute the validation loss
            val_running_loss += loss.item()                           # Accumulate the running loss

            # Calculate validation accuracy
            predicted = torch.round(outputs).squeeze()
            correct_val += (predicted == labels).sum().item()         # Compare between the predicted and actual
            total_val += labels.size(0)                               # Count total samples

    # Calculate average validation loss and accuracy for the epoch
    val_loss = val_running_loss / len(val_loader)   # Calculate average validation loss
    val_accuracy = correct_val / total_val          # Calculate validation accuracy

    # Update the learning rate
    scheduler.step(val_loss)

    # Check for early stopping
    if check_early_stop(val_loss):
        break

    # Save metrics
    train_losses.append(train_loss)           # Append training loss
    train_accuracies.append(train_accuracy)   # Append training accuracy
    val_losses.append(val_loss)               # Append validation loss
    val_accuracies.append(val_accuracy)        # Append validation accuracy

    # Print epoch results
    print(f"Epoch [{epoch + 1}/{num_epochs}], \
          Training Loss: {train_loss:.4f}, \
          Training Accuracy: {train_accuracy:.2%}, \
          Validation Loss: {val_losses[-1]:.4f}, \
          Validation Accuracy: {val_accuracy:.2%}")  # Print epoch results

Epoch [1/15],           Training Loss: 0.2406,           Training Accuracy: 89.81%,           Validation Loss: 0.1892,           Validation Accuracy: 90.89%
Epoch [2/15],           Training Loss: 0.1301,           Training Accuracy: 95.00%,           Validation Loss: 0.1561,           Validation Accuracy: 93.33%
Epoch [3/15],           Training Loss: 0.1451,           Training Accuracy: 94.14%,           Validation Loss: 0.1086,           Validation Accuracy: 95.33%
Epoch [4/15],           Training Loss: 0.1565,           Training Accuracy: 95.67%,           Validation Loss: 0.2568,           Validation Accuracy: 89.78%
Epoch [5/15],           Training Loss: 0.1211,           Training Accuracy: 95.10%,           Validation Loss: 0.2731,           Validation Accuracy: 93.33%
Epoch [6/15],           Training Loss: 0.0935,           Training Accuracy: 96.10%,           Validation Loss: 0.1283,           Validation Accuracy: 95.78%
Epoch [7/15],           Training Loss: 0.1103,           T

**[4.9]** **Testing:** Now it is time to test our model. Initiate the `model.eval()` along with `torch.no_grad()` to turn off the gradients. Finally calculate the total and correct value.

In [ ]:
# Solution
# Evaluate the model on the test set
correct = 0     # Initialize the number of correctly predicted samples on the test set
total = 0       # Initialize the total number of samples in the test set

with torch.no_grad():                                             # Disable gradient calculation for evaluation
    for images, labels in test_loader:                            # Iterate over batches of test data
        images, labels = images.to(device), labels.to(device)     # Move images and labels to the device
        outputs = vgg16(images)                                   # Forward pass: compute model outputs
        predicted = torch.round(outputs).squeeze()                # Round the predicted probabilities to obtain binary predictions
        total += labels.size(0)                                   # Count the total number of samples in the current batch
        correct += (predicted == labels).sum().item()             # Count the number of correct predictions

# Compute accuracy on the test set
test_accuracy = correct / total                         # Calculate the ratio of correct predictions to total samples
print(f'Accuracy on test set: {test_accuracy:.2%}')     # Print the accuracy on the test set

Accuracy on test set: 96.22%


By unfreezing the last four convolutional layers of the pre-trained VGG16 and training the model for 15 more epochs, we achieved even better accuracy. We have now 95% accuracy on the testing set.

### Discussion: What are the reasons why fine-tuning helped to improve the accuracy score.